#### Librerias

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import joblib
from scipy import sparse
from tqdm import tqdm

tqdm.pandas()

Mounted at /content/drive


#### Variables

In [ ]:
TRAIN_PATH = "/content/drive/MyDrive/Trabajo práctico 3/data/training.1600000.processed.noemoticon.csv"
TEST_PATH = "/content/drive/MyDrive/Trabajo práctico 3/data/testdata.manual.2009.06.14.csv"
MODELOS_PATH = "/content/drive/MyDrive/Trabajo práctico 3/modelos"

cols = ['polarity', 'id', 'date', 'query', 'user', 'text']
MAX_FEATURES = 20000

#### Lectura de datos

In [ ]:
df_train = pd.read_csv(TRAIN_PATH, encoding='latin-1', header=None, names=cols)
df_test = pd.read_csv(TEST_PATH, encoding='latin-1', header=None, names=cols)

df_train.shape, df_test.shape

((1600000, 6), (498, 6))

#### Función de limpieza

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def limpieza(texto):
    texto = texto.lower()
    texto = re.sub(r'http\S+|www\S+', '', texto)
    texto = re.sub(r'@\w+', '', texto)
    texto = re.sub(r'#', '', texto)
    texto = re.sub(r'[^a-z\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    tokens = [t for t in texto.split() if t not in stop_words]
    return ' '.join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cada paso de la limpieza tiene un motivo: las URLs y menciones (@usuario) se
eliminan porque son ruido sin carga de sentimiento; del hashtag se saca solo el
símbolo # pero se conserva la palabra (puede tener información útil, ej. #terrible);
y se eliminan las stopwords porque aparecen en casi todos los tweets sin importar
el sentimiento. Se usó regex + stopwords de nltk en lugar de spacy con lematización
por una cuestión de performance: con 1.6 millones de tweets, procesar fila por fila
con spacy tardaría demasiado, y en textos tan cortos la lematización aporta poco.

#### Aplicar limpieza

In [ ]:
df_train["text_limpio"] = df_train["text"].progress_apply(limpieza)
df_test["text_limpio"] = df_test["text"].progress_apply(limpieza)

100%|██████████| 498/498 [00:00<00:00, 33879.90it/s]


#### Chequeo de tweets vacíos tras la limpieza

In [ ]:
vacios_train = (df_train["text_limpio"].str.strip() == "").sum()
vacios_test = (df_test["text_limpio"].str.strip() == "").sum()

print(f"Tweets vacíos tras limpieza — train: {vacios_train} ({vacios_train/len(df_train)*100:.2f}%)")
print(f"Tweets vacíos tras limpieza — test: {vacios_test} ({vacios_test/len(df_test)*100:.2f}%)")

Tweets vacíos tras limpieza — train: 7711 (0.48%)
Tweets vacíos tras limpieza — test: 0 (0.00%)


Algunos tweets quedan vacíos después de limpiar: son los que eran solo una URL,
una mención o puras stopwords. Al ser un porcentaje mínimo, se dejan en el dataset
— no vale la pena complicar el pipeline por tan pocas filas.

#### Guardado de datos limpios

In [ ]:
df_train.to_pickle(f"{MODELOS_PATH}/train_limpio.pkl")
df_test.to_pickle(f"{MODELOS_PATH}/test_limpio.pkl")

#### Vectorización — Bag of Words

In [ ]:
vectorizer_bow = CountVectorizer(max_features=MAX_FEATURES)

X_train_bow = vectorizer_bow.fit_transform(df_train["text_limpio"])
X_test_bow = vectorizer_bow.transform(df_test["text_limpio"])

X_train_bow.shape, X_test_bow.shape

((1600000, 20000), (498, 20000))

#### Vectorización — TF-IDF

In [ ]:
vectorizer_tfidf = TfidfVectorizer(max_features=MAX_FEATURES)

X_train_tfidf = vectorizer_tfidf.fit_transform(df_train["text_limpio"])
X_test_tfidf = vectorizer_tfidf.transform(df_test["text_limpio"])

X_train_tfidf.shape, X_test_tfidf.shape

((1600000, 20000), (498, 20000))

Se generaron dos representaciones del mismo texto: Bag of Words (conteo simple de
palabras, para Naive Bayes) y TF-IDF (conteo ponderado por qué tan distintiva es
cada palabra, para Logistic Regression). En ambas se limitó el vocabulario a las
20.000 palabras más frecuentes (max_features): sin ese límite, con 1.6M de tweets
el vocabulario explota a cientos de miles de términos, la mayoría typos o palabras
que aparecen una sola vez y no aportan nada. Importante: el vectorizador se ajusta
(fit) solo con el training y al test únicamente se lo transforma — si se ajustara
con el test, estaríamos filtrando información que el modelo no debería conocer.

#### Guardado de vectorizadores y matrices

In [ ]:
joblib.dump(vectorizer_bow, f"{MODELOS_PATH}/vectorizer_bow.pkl")
joblib.dump(vectorizer_tfidf, f"{MODELOS_PATH}/vectorizer_tfidf.pkl")

sparse.save_npz(f"{MODELOS_PATH}/X_train_bow.npz", X_train_bow)
sparse.save_npz(f"{MODELOS_PATH}/X_test_bow.npz", X_test_bow)
sparse.save_npz(f"{MODELOS_PATH}/X_train_tfidf.npz", X_train_tfidf)
sparse.save_npz(f"{MODELOS_PATH}/X_test_tfidf.npz", X_test_tfidf)

np.save(f"{MODELOS_PATH}/y_train.npy", df_train["polarity"].values)
np.save(f"{MODELOS_PATH}/y_test.npy", df_test["polarity"].values)

Todo lo procesado (datos limpios, vectorizadores ajustados y matrices) queda
guardado en la carpeta modelos/ de Drive. Las notebooks siguientes cargan estos
archivos directamente en lugar de reprocesar — esto garantiza que todas trabajen
con exactamente la misma versión de los datos y evita repetir varios minutos de
cómputo cada vez.